# Evaluation and Holdout Results

This notebook summarizes the three-seed adaptation experiments and the frozen CIC holdout results. It checks the saved metrics and creates figures for the report. It does not train models or generate new predictions.

## 1. Load the results

Load the adaptation results, experiment settings, and saved holdout metrics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: change BASE to your local results folder.')

BASE = Path('/content/drive/MyDrive/datanew/a4_analysis')
RUN_DIR = BASE / 'reproduction_runs' / 'source_adaptation'
OUT_DIR = BASE / 'reproduction_runs' / 'evaluation_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ADAPTATION_RESULTS = RUN_DIR / 'adaptation_three_seed_results.csv'
RUN_CONFIG = RUN_DIR / 'run_config.json'
HOLDOUT_SUMMARY = BASE / 'cell_44R_holdout_summary.json'
CATEGORY_RESULTS = BASE / 'cell_44R_category_results.csv'
HOLDOUT_PREDICTIONS = BASE / 'cell_44R_holdout_predictions.csv'

required_files = [ADAPTATION_RESULTS, RUN_CONFIG, HOLDOUT_SUMMARY, CATEGORY_RESULTS]
missing = [str(p) for p in required_files if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing saved results. Check these paths:\n' + '\n'.join(missing))

print('Adaptation results:', ADAPTATION_RESULTS)
print('Frozen holdout summary:', HOLDOUT_SUMMARY)
print('Evaluation outputs:', OUT_DIR)

## 2. Compare adaptation methods

Compare random sampling, standard RL, and diversity-aware RL using the same 1,000-label budget. The table shows each seed and the average performance across three runs.

In [ ]:
with RUN_CONFIG.open('r', encoding='utf-8') as f:
    config = json.load(f)

results = pd.read_csv(ADAPTATION_RESULTS)
expected = {'Seed', 'Method', 'Macro_F1', 'Binary_PR_AUC',
            'Exact_Type_Accuracy', 'Reviewed_Attacks', 'Supported_Types'}
if not expected.issubset(results.columns):
    raise ValueError(f'Adaptation results missing: {sorted(expected - set(results.columns))}')

methods = ['Random', 'Standard_RL', 'Diversity_Aware_RL']
seeds = [51, 73, 97]
assert len(results) == 9, 'Expected three methods evaluated over three seeds.'
assert set(results['Method']) == set(methods)
assert set(results['Seed']) == set(seeds)
assert not results.duplicated(['Seed', 'Method']).any()
assert all(results[col].between(0, 1).all() for col in
           ['Macro_F1', 'Binary_PR_AUC', 'Exact_Type_Accuracy'])
assert int(config['label_budget']) == 1000, 'Check the label budget in notebook 02.'
assert config['independent_holdout_used'] is False

method_summary = (results.groupby('Method', as_index=False)
    .agg(Mean_Macro_F1=('Macro_F1', 'mean'),
         Mean_Binary_PR_AUC=('Binary_PR_AUC', 'mean'),
         Mean_Exact_Type_Accuracy=('Exact_Type_Accuracy', 'mean'),
         Mean_Reviewed_Attacks=('Reviewed_Attacks', 'mean'),
         Mean_Supported_Types=('Supported_Types', 'mean')))
method_summary = method_summary.set_index('Method').loc[methods].reset_index()

print('Three-seed development results:')
display(results.sort_values(['Seed', 'Method']).round(4))
print('Mean across three seeds:')
display(method_summary.round(4))

method_summary.to_csv(OUT_DIR / 'adaptation_three_seed_means.csv', index=False)

## 3. Frozen holdout results

The adapted classifier was evaluated on 20,000 previously unused CIC records. The records came from the same CIC dataset, so this is not an independent cross-dataset test. This section summarizes the saved results and recall for each attack type.

In [ ]:
with HOLDOUT_SUMMARY.open('r', encoding='utf-8') as f:
    holdout = json.load(f)

category_df = pd.read_csv(CATEGORY_RESULTS)
summary_fields = [
    'Holdout_Records', 'Attack_Records', 'Attack_Prevalence',
    'Multiclass_Macro_F1', 'Exact_Attack_Type_Accuracy',
    'Correct_Attack_Type_Predictions', 'Binary_PR_AUC',
    'Binary_ROC_AUC', 'Alert_Budget_Fraction', 'Alert_Budget_Records',
    'Attacks_Found_At_Budget', 'Precision_At_Budget', 'Recall_At_Budget',
    'Predefined_Attack_Types'
]
missing_fields = [name for name in summary_fields if name not in holdout]
if missing_fields:
    raise ValueError(f'Holdout summary missing: {missing_fields}')

required_category_cols = {'Attack_Category', 'Test_Records', 'Recall', 'Precision', 'F1'}
if not required_category_cols.issubset(category_df.columns):
    raise ValueError('Category-results file has unexpected columns.')

assert int(holdout['Holdout_Records']) == 20000
assert int(holdout['Attack_Records']) == 2417
assert int(holdout['Correct_Attack_Type_Predictions']) == 2218
assert np.isclose(float(holdout['Attack_Prevalence']), 2417 / 20000)
assert np.isclose(float(holdout['Exact_Attack_Type_Accuracy']), 2218 / 2417)
assert int(holdout['Alert_Budget_Records']) == 2500
assert int(category_df['Test_Records'].sum()) == 2417
assert category_df['Attack_Category'].is_unique
assert category_df[['Precision', 'Recall', 'F1']].apply(
    lambda col: col.between(0, 1).all()).all()
assert holdout.get('Model_Retrained') is False
assert holdout.get('Test_Labels_Used_For_Fitting') is False

headline = pd.DataFrame({
    'Metric': ['Records', 'Attacks', 'Multiclass macro-F1',
               'Exact attack-type accuracy', 'Binary PR-AUC', 'Binary ROC-AUC',
               'Precision at 12.5% alert budget', 'Recall at 12.5% alert budget'],
    'Value': [holdout['Holdout_Records'], holdout['Attack_Records'],
              holdout['Multiclass_Macro_F1'], holdout['Exact_Attack_Type_Accuracy'],
              holdout['Binary_PR_AUC'], holdout['Binary_ROC_AUC'],
              holdout['Precision_At_Budget'], holdout['Recall_At_Budget']]
})

print('Saved frozen holdout evaluation:')
display(headline)
print('Results by actual attack category:')
display(category_df.sort_values('Recall').round(4))

headline.to_csv(OUT_DIR / 'frozen_holdout_key_results.csv', index=False)
category_df.to_csv(OUT_DIR / 'frozen_holdout_category_results.csv', index=False)

## 4. Check holdout metrics

When saved record-level predictions are available, use them to recalculate the main metrics and compare them with the summary. This check does not run the classifier again.

In [ ]:
if not HOLDOUT_PREDICTIONS.exists():
    print('Row-level prediction file not available. Saved summary and category table were checked.')
else:
    predictions = pd.read_csv(HOLDOUT_PREDICTIONS)
    cols = {'True_Binary_Label', 'True_Attack_Category',
            'Predicted_Category', 'Attack_Score'}
    if not cols.issubset(predictions.columns):
        raise ValueError('Saved prediction file has unexpected columns.')

    y = predictions['True_Binary_Label'].to_numpy(dtype=int)
    true_cat = predictions['True_Attack_Category'].astype(str).to_numpy()
    pred_cat = predictions['Predicted_Category'].astype(str).to_numpy()
    scores = predictions['Attack_Score'].to_numpy(dtype=float)

    assert len(y) == 20000 and set(np.unique(y)).issubset({0, 1})
    assert int(y.sum()) == 2417 and np.isfinite(scores).all()
    assert np.all((scores >= 0) & (scores <= 1))
    assert np.all(true_cat[y == 0] == 'Benign')

    ap = average_precision_score(y, scores)
    auc = roc_auc_score(y, scores)
    accurate_types = int(np.sum((pred_cat == true_cat) & (y == 1)))

    # Use stable sorting when attack scores are tied.
    top = np.argsort(-scores, kind='stable')[:2500]
    found = int(y[top].sum())

    checks = {
        'Binary_PR_AUC': ap,
        'Binary_ROC_AUC': auc,
        'Exact_Attack_Type_Accuracy': accurate_types / int(y.sum()),
        'Precision_At_Budget': found / 2500,
        'Recall_At_Budget': found / int(y.sum()),
    }
    for name, value in checks.items():
        if not np.isclose(value, float(holdout[name]), rtol=1e-8, atol=1e-10):
            raise AssertionError(f'Saved predictions do not match {name}.')

    assert accurate_types == int(holdout['Correct_Attack_Type_Predictions'])
    assert found == int(holdout['Attacks_Found_At_Budget'])

    # Calculate macro-F1 only when the full category list is available.
    predefined_n = int(holdout['Predefined_Attack_Types'])
    new_categories = holdout.get('New_Holdout_Attack_Types', [])
    if len(category_df) == predefined_n and not new_categories:
        fixed_labels = category_df['Attack_Category'].astype(str).tolist()
        recalculated_f1 = f1_score(
            true_cat, pred_cat, labels=fixed_labels,
            average='macro', zero_division=0
        )
        assert np.isclose(recalculated_f1, float(holdout['Multiclass_Macro_F1']))
        checks['Multiclass_Macro_F1'] = recalculated_f1
    else:
        print('Macro-F1 not recalculated: exact fixed taxonomy not available in saved table.')

    print('Saved predictions match the holdout summary:')
    display(pd.DataFrame(checks.items(), columns=['Metric', 'Verified_Value']))
    del predictions, y, true_cat, pred_cat, scores

## 5. Visualize the results

The first panel compares the adaptation methods across three seeds. The second shows recall for individual attack categories on the CIC holdout. Larger points represent categories with more records.

In [ ]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'svg.fonttype': 'none', 'pdf.fonttype': 42
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.4),
                              gridspec_kw={'width_ratios': [1, 1.3]})
fig.patch.set_facecolor('white')

colors = {'Random': '#8391a3', 'Standard_RL': '#3c77ad',
          'Diversity_Aware_RL': '#c65b45'}
pretty = {'Random': 'Random', 'Standard_RL': 'Standard RL',
          'Diversity_Aware_RL': 'Diversity-aware RL'}

for row, method in enumerate(methods):
    vals = results.loc[results['Method'] == method, 'Macro_F1'].to_numpy()
    ax1.scatter(vals, np.full(len(vals), row), s=68,
                color=colors[method], alpha=.65, zorder=3)
    ax1.scatter([vals.mean()], [row], marker='D', s=130,
                color=colors[method], edgecolors='white', linewidths=1.1, zorder=4)
    ax1.text(float(vals.mean()) + .012, row - .19, f'{vals.mean():.3f}',
             color=colors[method], weight='bold', fontsize=10)
ax1.set_yticks(range(len(methods)), [pretty[m] for m in methods])
ax1.invert_yaxis()
ax1.set_xlim(0, max(.7, float(results['Macro_F1'].max()) + .08))
ax1.set_xlabel('Multiclass macro-F1')
ax1.set_title('(a) CIC development | three selection seeds', loc='left',
              weight='bold', fontsize=11)
ax1.grid(axis='x', color='#e2e8ef', alpha=.8)
ax1.set_axisbelow(True)

cat = category_df.sort_values('Recall', ascending=True).reset_index(drop=True)
support = cat['Test_Records'].to_numpy(dtype=float)
sizes = 42 + 310 * np.sqrt(support / max(float(support.max()), 1.0))
y_pos = np.arange(len(cat))
ax2.scatter(cat['Recall'], y_pos, s=sizes, c=cat['Recall'],
            vmin=0, vmax=1, cmap='viridis', edgecolors='#2f4358',
            linewidth=.55, alpha=.90, zorder=3)
ax2.set_yticks(y_pos, cat['Attack_Category'])
ax2.set_xlim(-.05, 1.07)
ax2.set_xlabel('Attack-type recall')
ax2.set_title('(b) Frozen CIC holdout | attack-type recall', loc='left',
              weight='bold', fontsize=11)
ax2.grid(axis='x', color='#e2e8ef', alpha=.8)
ax2.set_axisbelow(True)

fig.suptitle('CSC786 | Adaptation and frozen holdout evaluation',
             fontsize=15, weight='bold', x=.03, ha='left', y=.99)
fig.text(.03, .025,
         'Diamonds = three-seed mean; small dots = individual seeds. '
         'Holdout bubble size = number of attack records. '
         'Holdout is from the same CIC corpus.',
         fontsize=9, color='#526173')
fig.subplots_adjust(left=.19, right=.97, bottom=.13, top=.85, wspace=.55)

stem = OUT_DIR / 'CSC786_Fig03_Evaluation_Overview'
for ext in ['png', 'svg', 'pdf']:
    fig.savefig(f'{stem}.{ext}', dpi=300 if ext == 'png' else None,
                facecolor='white')
plt.show()
print('Figure saved:', stem)